## 3D synthetic loss grid (appendix figures)

`Run All` writes into `img_results/`:

- `loss_p{0.9,0.8,0.7,0.5}_sigma{0.0,0.5,1.0,2.0}.pdf`: the 16 loss-dynamics panels.
- `delta_conf_sigma{0.0,0.5,1.0,2.0}.pdf`: the 4 $\Delta_{\mathrm{conf}}$ panels (4-point, p in {0.9, 0.8, 0.7, 0.5}, matching the loss grid).


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


def make_synthetic_dataset(n=10000, p_spurious=0.8, sigma_small=0.0, seed=0):
    rng = np.random.RandomState(seed)
    z = rng.uniform(-1.0, 1.0, size=n)
    x1_core = np.sin(5 * z) + np.exp(z)              # core feature; label = sign(x1_core)
    y = np.sign(x1_core)
    y01 = (y > 0).astype(np.float32)

    u = np.sin(5 * z)
    v = np.exp(z)
    n_spurious = int(p_spurious * n)
    indices = np.arange(n)
    rng.shuffle(indices)
    is_spurious_example = np.zeros(n, dtype=bool)
    is_spurious_example[indices[:n_spurious]] = True

    x2 = np.empty(n, dtype=np.float32)
    x3 = np.empty(n, dtype=np.float32)
    for i in range(n):
        eps2 = rng.normal(0.0, sigma_small)
        eps3 = rng.normal(0.0, sigma_small)
        if is_spurious_example[i]:                   # spurious: sign(x2 + x3) agrees with y
            x2[i] = u[i] + eps2
            x3[i] = v[i] + eps3
        else:                                        # non-spurious: flip so sign(x2 + x3) opposes y
            x2[i] = -(u[i] + eps2)
            x3[i] = -(v[i] + eps3)

    x1 = np.tanh(x1_core).astype(np.float32)         # causal feature shown to the model
    X = np.stack([x1, x2, x3], axis=1).astype(np.float32)

    # near-boundary vs non-boundary by causal margin (smallest / largest 20%)
    abs_margin = np.abs(y * x1_core)
    is_confusing = abs_margin <= np.percentile(abs_margin, 20)
    is_nonconfusing = abs_margin >= np.percentile(abs_margin, 80)

    # spurious vs non-spurious by sign(x2 + x3)
    spur_sign = np.sign(x2 + x3)
    spur_sign[spur_sign == 0] = 1
    spur_agrees = spur_sign == y

    groups = {
        "conf_spur": np.where(is_confusing & spur_agrees)[0],
        "conf_nonspur": np.where(is_confusing & ~spur_agrees)[0],
        "nonconf_spur": np.where(is_nonconfusing & spur_agrees)[0],
        "nonconf_nonspur": np.where(is_nonconfusing & ~spur_agrees)[0],
    }
    return X, y01, groups


class LogisticReg(nn.Module):
    def __init__(self, input_dim=3):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


def run_experiment(n=10000, p_spurious=0.8, sigma_small=0.0, batch_size=256,
                   num_epochs=10, lr=1e-1, seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    X, y01, groups = make_synthetic_dataset(n=n, p_spurious=p_spurious,
                                            sigma_small=sigma_small, seed=seed)

    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y01)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    model = LogisticReg(input_dim=3)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    masks = {}
    for name, idxs in groups.items():
        mask = torch.zeros(X_t.size(0), dtype=torch.bool)
        mask[idxs] = True
        masks[name] = mask

    loss_history = {name: [] for name in groups}
    loss_history["all"] = []

    model.eval()
    with torch.no_grad():
        losses0 = criterion(model(X_t), y_t)
    init_group_loss = {}
    for name, mask in masks.items():
        init_group_loss[name] = losses0[mask].mean().item() if mask.sum() > 0 else float("nan")
    loss_history["all"].append(losses0.mean().item())
    for name in groups:
        loss_history[name].append(init_group_loss[name])

    delta_conf_history = []
    for epoch in range(num_epochs):
        model.train()
        for xb, yb in loader:
            loss = criterion(model(xb), yb).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            losses_full = criterion(model(X_t), y_t)
            loss_history["all"].append(losses_full.mean().item())
            current_group_loss = {}
            for name, mask in masks.items():
                gl = losses_full[mask].mean().item() if mask.sum() > 0 else float("nan")
                loss_history[name].append(gl)
                current_group_loss[name] = gl
            delta_conf = ((current_group_loss["conf_nonspur"] - init_group_loss["conf_nonspur"])
                          - (current_group_loss["conf_spur"] - init_group_loss["conf_spur"]))
            delta_conf_history.append(delta_conf)

    return loss_history, delta_conf_history


In [ ]:
import os
import matplotlib.ticker as mticker

os.makedirs("img_results", exist_ok=True)

p_list = [0.9, 0.8, 0.7, 0.5]
sigma_list = [0.0, 0.5, 1.0, 2.0]
delta_conf_grid = {sigma: [] for sigma in sigma_list}

for sigma in sigma_list:
    # PASS 1: train every p in this row, collect histories for a shared row y-limit
    sigma_data = {}
    for p in p_list:
        history, delta_conf_history = run_experiment(
            n=10000, p_spurious=p, sigma_small=sigma,
            batch_size=128, num_epochs=10, lr=0.5, seed=0)
        epochs = np.arange(len(history["all"]))
        sigma_data[p] = (history, delta_conf_history, epochs)
        print(f"trained p={p} sigma={sigma}")

    plot_keys = ("conf_nonspur", "conf_spur", "nonconf_nonspur", "nonconf_spur")
    row_ymax = 0.0
    for p in p_list:
        history, _, _ = sigma_data[p]
        for key in plot_keys:
            values = np.asarray(history[key], dtype=float)
            if values.size == 0:
                raise ValueError(f"Empty loss history for sigma={sigma}, p={p}, key={key}")
            if not np.all(np.isfinite(values)):
                raise ValueError(f"Non-finite loss for sigma={sigma}, p={p}, key={key}")
            row_ymax = max(row_ymax, float(np.max(values)))
    row_ymax = max(row_ymax * 1.05, 1e-6)

    # PASS 2: plot each panel with the shared row y-limit (share-axes 4x4 grid layout).
    # Only the bottom row shows x labels/ticks and only the left column shows y labels/ticks;
    # inner panels use a narrower/shorter canvas so the grid tiles without gaps.
    for p in p_list:
        history, delta_conf_history, epochs = sigma_data[p]
        show_xlabel = (sigma == sigma_list[-1])
        show_ylabel = (p == p_list[0])

        TICK_FONT = 17
        LABEL_FONT = 19
        LEGEND_FONT = 11
        if show_ylabel:
            panel_width_in = 4.13
            plot_left_in = 0.85
        else:
            panel_width_in = 3.28
            plot_left_in = 0.06
        plot_right_in = panel_width_in - 0.04
        plot_top_in = 0.16
        plot_height_in = 3.16
        plot_bottom_in = 0.64 if show_xlabel else 0.16
        panel_height_in = plot_bottom_in + plot_height_in + plot_top_in

        fig = plt.figure(figsize=(panel_width_in, panel_height_in))
        ax = fig.add_axes([
            plot_left_in / panel_width_in,
            plot_bottom_in / panel_height_in,
            (plot_right_in - plot_left_in) / panel_width_in,
            plot_height_in / panel_height_in,
        ])
        plt.sca(ax)

        l1, = ax.plot(epochs, history["conf_nonspur"], linewidth=2.15, color="#3f8fd5", linestyle="-")
        l2, = ax.plot(epochs, history["conf_spur"], linewidth=2.15, color="#ff9933", linestyle="-")
        l3, = ax.plot(epochs, history["nonconf_nonspur"], linewidth=2.15, color="#009e73", linestyle="-")
        l4, = ax.plot(epochs, history["nonconf_spur"], linewidth=2.15, color="#b07adf", linestyle="-")

        ax.set_ylim(-row_ymax * 0.05, row_ymax)
        ax.set_xlim(-0.5, len(epochs) - 0.5)

        # one shared legend, on the top-right panel only
        if sigma == sigma_list[0] and p == p_list[-1]:
            ax.legend(
                [l1, l2, l3, l4],
                ["Near-Boundary & Non-spurious", "Near-Boundary & Spurious",
                 "Non-Boundary & Non-spurious", "Non-Boundary & Spurious"],
                loc="upper right", bbox_to_anchor=(0.995, 0.995), fontsize=LEGEND_FONT,
                frameon=True, framealpha=0.85, borderpad=0.3, labelspacing=0.3,
                handlelength=2.0, handletextpad=0.5, borderaxespad=0.15,
            )

        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))
        ax.tick_params(axis="both", which="major", labelsize=TICK_FONT)

        if show_xlabel:
            plt.xlabel("Training epochs", fontsize=LABEL_FONT)
        else:
            ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
        if show_ylabel:
            plt.ylabel("Average training loss", fontsize=LABEL_FONT)
        else:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

        fig.savefig(f"img_results/loss_p{p}_sigma{sigma}.pdf")
        plt.close()

        delta_conf_grid[sigma].append(delta_conf_history[0] if delta_conf_history else float("nan"))

# delta_conf vs p, one panel per sigma
for sigma in sigma_list:
    plt.figure(figsize=(4, 4))
    plt.plot(p_list, delta_conf_grid[sigma], marker="o")
    plt.xlabel(r"$p_{\mathrm{sp}}$", fontsize=19)
    plt.ylabel(r"$\Delta_{\mathrm{conf}}$", fontsize=19)
    ax = plt.gca()
    ax.set_xticks(p_list)
    ax.invert_xaxis()
    ax.tick_params(axis="both", which="major", labelsize=17)
    plt.tight_layout()
    plt.savefig(f"img_results/delta_conf_sigma{sigma}.pdf", bbox_inches="tight")
    plt.close()

print("wrote 16 loss panels + 4 delta_conf panels into img_results/")
